# Guitar Pedalboard One Cell

A single-cell pedalboard UI for the live PYNQ-Z2 effect chain.
Built on top of `GuitarEffectSwitcher.ipynb` but consolidates every
effect into one ipywidgets panel, and drives the new pedal-mask
distortion API instead of the legacy `distortion=`/`distortion_*`
knobs.

## Effect chain

```
Noise Gate -> Overdrive -> Distortion Pedalboard
             -> Amp Simulator -> Cab IR -> EQ -> Reverb
```

## Distortion pedals

- **Implemented in the current bitstream**: `clean_boost`,
  `tube_screamer`, `rat` (mapped onto the existing RAT stage),
  `metal`.
- **Reserved (bypass stub)**: `ds1`, `big_muff`, `fuzz_face`.
  Selecting them sets the corresponding pedal-mask bit and prints a
  warning, but the FPGA stage currently leaves audio bit-exact.

## Loud-output warning

- Drop headphone volume **before** clicking Apply.
- Start at `level = 30..35`. Crank only after you have heard the
  preset at low volume.
- `rat` and `metal` are very high-gain. Stacking pedals
  (`exclusive=False`) is louder still — the cell auto-trims `level`
  to 25 when you turn stack mode on.

## ADC HPF default-on

`config_codec()` enables the ADAU1761 ADC digital HPF, so after
load `R19_ADC_CONTROL == 0x23`. That filter is a ~2 Hz DC blocker,
**not** a 20-40 Hz guitar low-cut. The status panel below confirms
the value on every Apply / Refresh.

## What this notebook does *not* change

Notebook-only update. No bitstream rebuild needed; the active
bit/hwh on the board is the pedal-mask build at
`baa97ff` and remains untouched.


In [ ]:
import sys
import traceback
from pathlib import Path

PROJECT_ROOT = "/home/xilinx/Audio-Lab-PYNQ"
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import ipywidgets as widgets
from IPython.display import display, clear_output

from audio_lab_pynq.AudioLabOverlay import AudioLabOverlay

# ---- output console (used everywhere below) -------------------------
status = widgets.Output(layout=widgets.Layout(border="1px solid #ccc",
                                              max_height="320px",
                                              overflow="auto"))

# ---- reserved-pedal helper ------------------------------------------
PEDAL_LABEL_TO_API = {
    "off":                 None,
    "clean_boost":         "clean_boost",
    "tube_screamer":       "tube_screamer",
    "rat":                 "rat",
    "metal":               "metal",
    "ds1_reserved":        "ds1",
    "big_muff_reserved":   "big_muff",
    "fuzz_face_reserved":  "fuzz_face",
}
RESERVED_PEDALS = {"ds1", "big_muff", "fuzz_face"}

# ---- overlay initialisation -----------------------------------------
ovl = None
try:
    ovl = AudioLabOverlay()
    # Always start from a known-quiet state.
    ovl.clear_distortion_pedals()
    ovl.set_distortion_settings(drive=20, tone=50, level=35,
                                bias=50, tight=50, mix=100)
    ovl.set_guitar_effects(distortion_on=False)
except Exception:
    with status:
        print("AudioLabOverlay() failed:")
        traceback.print_exc()

def _say(*args):
    with status:
        print(*args)

# ---- helpers --------------------------------------------------------
def refresh_status(*_):
    if ovl is None:
        return
    with status:
        clear_output()
        try:
            print("ADC HPF        :", ovl.codec.get_adc_hpf_state())
            print("R19_ADC_CONTROL:", hex(ovl.codec.R19_ADC_CONTROL[0]))
            s = ovl.get_distortion_settings()
            print("pedal_mask     : 0x{:02x}".format(s["pedal_mask"]))
            print("active pedals  :",
                  [k for k, v in s["pedals"].items() if v] or "(none)")
            print("dist params    : drive={drive} tone={tone} level={level}"
                  " bias={bias} tight={tight} mix={mix}".format(**s))
        except Exception:
            traceback.print_exc()

def _selected_pedals(stack_mode, dropdown_value, multiselect_value):
    """Translate the dropdown / multi-select widgets into the list of
    API pedal names to enable. Returns (pedal_names, warning_lines)."""
    warnings = []
    if stack_mode:
        names = []
        for label in multiselect_value:
            api = PEDAL_LABEL_TO_API[label]
            if api is None:
                continue
            names.append(api)
            if api in RESERVED_PEDALS:
                warnings.append("note: {} is reserved; FPGA stage is a "
                                "bypass stub in the current bitstream"
                                .format(api))
        return names, warnings
    api = PEDAL_LABEL_TO_API[dropdown_value]
    if api is None:
        return [], warnings
    if api in RESERVED_PEDALS:
        warnings.append("note: {} is reserved; FPGA stage is a bypass "
                        "stub in the current bitstream".format(api))
    return [api], warnings

def apply_settings(*_):
    if ovl is None:
        _say("overlay not initialised; nothing to apply")
        return
    try:
        with status:
            clear_output()

        # ---- gather UI state -------------------------------------------------
        master_on = w_dist_master.value
        stack_mode = w_dist_stack.value
        exclusive = w_dist_exclusive.value and not stack_mode
        pedal_names, warns = _selected_pedals(stack_mode,
                                              w_dist_pedal.value,
                                              w_dist_multi.value)
        for line in warns:
            _say(line)

        if stack_mode:
            # Auto-attenuate when stacking high-gain pedals together.
            if w_dist_level.value > 25:
                _say("stack mode: auto-trimming distortion level "
                     "{} -> 25 to avoid loud surprises".format(
                         w_dist_level.value))
                w_dist_level.value = 25

        # ---- write distortion section ---------------------------------------
        if not master_on or not pedal_names:
            ovl.clear_distortion_pedals()
            ovl.set_distortion_settings(
                drive=w_dist_drive.value, tone=w_dist_tone.value,
                level=w_dist_level.value, bias=w_dist_bias.value,
                tight=w_dist_tight.value, mix=w_dist_mix.value,
            )
        else:
            if exclusive:
                ovl.set_distortion_settings(
                    pedal=pedal_names[0], exclusive=True,
                    drive=w_dist_drive.value, tone=w_dist_tone.value,
                    level=w_dist_level.value, bias=w_dist_bias.value,
                    tight=w_dist_tight.value, mix=w_dist_mix.value,
                )
            else:
                ovl.clear_distortion_pedals()
                ovl.set_distortion_pedals(**{p: True for p in pedal_names})
                ovl.set_distortion_settings(
                    drive=w_dist_drive.value, tone=w_dist_tone.value,
                    level=w_dist_level.value, bias=w_dist_bias.value,
                    tight=w_dist_tight.value, mix=w_dist_mix.value,
                )

        # ---- write the rest of the chain via the legacy API -----------------
        words = ovl.set_guitar_effects(
            noise_gate_on=w_gate_on.value,
            noise_gate_threshold=w_gate_thr.value,
            overdrive_on=w_od_on.value,
            overdrive_drive=w_od_drive.value,
            overdrive_tone=w_od_tone.value,
            overdrive_level=w_od_level.value,
            distortion_on=master_on,
            amp_on=w_amp_on.value,
            amp_input_gain=w_amp_gain.value,
            amp_bass=w_amp_bass.value,
            amp_middle=w_amp_mid.value,
            amp_treble=w_amp_treble.value,
            amp_presence=w_amp_presence.value,
            amp_resonance=w_amp_resonance.value,
            amp_master=w_amp_master.value,
            amp_character=w_amp_character.value,
            cab_on=w_cab_on.value,
            cab_mix=w_cab_mix.value,
            cab_level=w_cab_level.value,
            cab_model=w_cab_model.value,
            cab_air=w_cab_air.value,
            eq_on=w_eq_on.value,
            eq_low=w_eq_low.value,
            eq_mid=w_eq_mid.value,
            eq_high=w_eq_high.value,
            reverb_on=w_rev_on.value,
            reverb_decay=w_rev_decay.value,
            reverb_tone=w_rev_tone.value,
            reverb_mix=w_rev_mix.value,
        )

        active = [n for n, on in [
            ("Noise Gate", w_gate_on.value),
            ("Overdrive", w_od_on.value),
            ("Distortion Pedalboard", master_on and bool(pedal_names)),
            ("Amp", w_amp_on.value),
            ("Cab IR", w_cab_on.value),
            ("EQ", w_eq_on.value),
            ("Reverb", w_rev_on.value),
        ] if on]
        _say("applied; active stages:", " -> ".join(active) or "(passthrough)")
        if pedal_names:
            _say("distortion pedals:", pedal_names)
        refresh_status()
    except Exception:
        with status:
            traceback.print_exc()

def safe_bypass(*_):
    if ovl is None:
        _say("overlay not initialised; nothing to bypass")
        return
    try:
        with status:
            clear_output()
        ovl.clear_distortion_pedals()
        ovl.set_distortion_settings(drive=20, tone=50, level=35,
                                    bias=50, tight=50, mix=100)
        ovl.set_guitar_effects(
            noise_gate_on=False,
            overdrive_on=False,
            distortion_on=False,
            rat_on=False,
            amp_on=False,
            cab_on=False,
            eq_on=False,
            reverb_on=False,
        )
        # Reset the UI checkboxes / sliders to the safe defaults too.
        w_gate_on.value = False
        w_od_on.value = False
        w_dist_master.value = False
        w_dist_pedal.value = "off"
        w_dist_multi.value = ()
        w_dist_stack.value = False
        w_dist_exclusive.value = True
        w_dist_drive.value = 20
        w_dist_tone.value = 50
        w_dist_level.value = 35
        w_dist_bias.value = 50
        w_dist_tight.value = 50
        w_dist_mix.value = 100
        w_amp_on.value = False
        w_cab_on.value = False
        w_eq_on.value = False
        w_rev_on.value = False
        _say("safe bypass complete")
        refresh_status()
    except Exception:
        with status:
            traceback.print_exc()

PRESETS = {
    "Clean Boost": dict(distortion_on=True, pedal="clean_boost",
                        drive=35, tone=50, level=45,
                        bias=50, tight=50, mix=100),
    "Tube Screamer Crunch": dict(distortion_on=True, pedal="tube_screamer",
                                 drive=45, tone=55, level=35,
                                 bias=50, tight=60, mix=100),
    "RAT Distortion": dict(distortion_on=True, pedal="rat",
                           drive=55, tone=45, level=35,
                           bias=50, tight=50, mix=100),
    "Metal Tight": dict(distortion_on=True, pedal="metal",
                        drive=55, tone=55, level=30,
                        bias=50, tight=75, mix=100),
}

def apply_preset(name):
    spec = PRESETS[name]
    w_dist_master.value = bool(spec.get("distortion_on", True))
    w_dist_stack.value = False
    w_dist_exclusive.value = True
    w_dist_pedal.value = spec["pedal"]
    w_dist_drive.value = spec["drive"]
    w_dist_tone.value = spec["tone"]
    w_dist_level.value = spec["level"]
    w_dist_bias.value = spec["bias"]
    w_dist_tight.value = spec["tight"]
    w_dist_mix.value = spec["mix"]
    apply_settings()

# ---- widgets --------------------------------------------------------
slider_layout = widgets.Layout(width="380px")
style = {"description_width": "120px"}

# Noise gate
w_gate_on = widgets.Checkbox(value=False, description="Noise Gate", indent=False)
w_gate_thr = widgets.IntSlider(value=8, min=0, max=100,
                               description="Threshold",
                               style=style, layout=slider_layout)

# Overdrive
w_od_on = widgets.Checkbox(value=False, description="Overdrive", indent=False)
w_od_drive = widgets.IntSlider(value=30, min=0, max=100, description="Drive",
                               style=style, layout=slider_layout)
w_od_tone = widgets.IntSlider(value=65, min=0, max=100, description="Tone",
                              style=style, layout=slider_layout)
w_od_level = widgets.IntSlider(value=100, min=0, max=200, description="Level",
                               style=style, layout=slider_layout)

# Distortion pedalboard
w_dist_master = widgets.Checkbox(value=False, description="Distortion master",
                                 indent=False)
w_dist_pedal = widgets.Dropdown(
    options=["off", "clean_boost", "tube_screamer", "rat", "metal",
             "ds1_reserved", "big_muff_reserved", "fuzz_face_reserved"],
    value="off", description="Pedal",
    style=style, layout=widgets.Layout(width="380px"))
w_dist_exclusive = widgets.Checkbox(value=True, description="Exclusive",
                                    indent=False)
w_dist_stack = widgets.Checkbox(value=False, description="Stack mode (advanced)",
                                indent=False)
w_dist_multi = widgets.SelectMultiple(
    options=["clean_boost", "tube_screamer", "rat", "metal",
             "ds1_reserved", "big_muff_reserved", "fuzz_face_reserved"],
    value=(), description="Stacked",
    style=style, layout=widgets.Layout(width="380px", height="120px"))
w_dist_drive = widgets.IntSlider(value=20, min=0, max=100, description="Drive",
                                 style=style, layout=slider_layout)
w_dist_tone = widgets.IntSlider(value=50, min=0, max=100, description="Tone",
                                style=style, layout=slider_layout)
w_dist_level = widgets.IntSlider(value=35, min=0, max=100, description="Level",
                                 style=style, layout=slider_layout)
w_dist_bias = widgets.IntSlider(value=50, min=0, max=100, description="Bias",
                                style=style, layout=slider_layout)
w_dist_tight = widgets.IntSlider(value=50, min=0, max=100, description="Tight",
                                 style=style, layout=slider_layout)
w_dist_mix = widgets.IntSlider(value=100, min=0, max=100, description="Mix",
                               style=style, layout=slider_layout)

# Amp simulator
w_amp_on = widgets.Checkbox(value=False, description="Amp Simulator",
                            indent=False)
w_amp_gain = widgets.IntSlider(value=35, min=0, max=100, description="Gain",
                               style=style, layout=slider_layout)
w_amp_bass = widgets.IntSlider(value=50, min=0, max=100, description="Bass",
                               style=style, layout=slider_layout)
w_amp_mid = widgets.IntSlider(value=50, min=0, max=100, description="Middle",
                              style=style, layout=slider_layout)
w_amp_treble = widgets.IntSlider(value=50, min=0, max=100, description="Treble",
                                 style=style, layout=slider_layout)
w_amp_presence = widgets.IntSlider(value=45, min=0, max=100,
                                   description="Presence",
                                   style=style, layout=slider_layout)
w_amp_resonance = widgets.IntSlider(value=35, min=0, max=100,
                                    description="Resonance",
                                    style=style, layout=slider_layout)
w_amp_master = widgets.IntSlider(value=80, min=0, max=150,
                                 description="Master",
                                 style=style, layout=slider_layout)
w_amp_character = widgets.IntSlider(value=35, min=0, max=100,
                                    description="Character",
                                    style=style, layout=slider_layout)

# Cab IR
w_cab_on = widgets.Checkbox(value=False, description="Cab IR", indent=False)
w_cab_mix = widgets.IntSlider(value=100, min=0, max=100, description="Mix",
                              style=style, layout=slider_layout)
w_cab_level = widgets.IntSlider(value=100, min=0, max=150, description="Level",
                                style=style, layout=slider_layout)
w_cab_model = widgets.Dropdown(
    options=[("1x12 open back", 0), ("2x12 british", 1), ("4x12 closed", 2)],
    value=1, description="Model",
    style=style, layout=widgets.Layout(width="380px"))
w_cab_air = widgets.IntSlider(value=50, min=0, max=100, description="Air",
                              style=style, layout=slider_layout)

# EQ
w_eq_on = widgets.Checkbox(value=False, description="EQ", indent=False)
w_eq_low = widgets.IntSlider(value=100, min=0, max=200, description="Low",
                             style=style, layout=slider_layout)
w_eq_mid = widgets.IntSlider(value=100, min=0, max=200, description="Mid",
                             style=style, layout=slider_layout)
w_eq_high = widgets.IntSlider(value=100, min=0, max=200, description="High",
                              style=style, layout=slider_layout)

# Reverb
w_rev_on = widgets.Checkbox(value=False, description="Reverb", indent=False)
w_rev_decay = widgets.IntSlider(value=30, min=0, max=100, description="Decay",
                                style=style, layout=slider_layout)
w_rev_tone = widgets.IntSlider(value=65, min=0, max=100, description="Tone",
                               style=style, layout=slider_layout)
w_rev_mix = widgets.IntSlider(value=20, min=0, max=100, description="Mix",
                              style=style, layout=slider_layout)

# Top-row buttons
b_apply = widgets.Button(description="Apply", button_style="primary")
b_bypass = widgets.Button(description="Safe Bypass", button_style="warning")
b_refresh = widgets.Button(description="Refresh Status")
b_apply.on_click(apply_settings)
b_bypass.on_click(safe_bypass)
b_refresh.on_click(refresh_status)

preset_buttons = []
for name in PRESETS:
    btn = widgets.Button(description=name)
    btn.on_click(lambda _b, n=name: apply_preset(n))
    preset_buttons.append(btn)

# Toggle Distortion stack-mode UI: hide single dropdown when stacking.
def _on_stack_change(change):
    stacking = bool(change["new"])
    w_dist_pedal.layout.display = "none" if stacking else None
    w_dist_exclusive.layout.display = "none" if stacking else None
    w_dist_multi.layout.display = None if stacking else "none"
w_dist_stack.observe(_on_stack_change, names="value")
_on_stack_change({"new": w_dist_stack.value})

# ---- layout ---------------------------------------------------------
title = widgets.HTML(
    "<h2 style='margin-bottom:4px'>Guitar Pedalboard One Cell</h2>"
    "<div style='color:#666;margin-bottom:8px'>"
    "Noise Gate -> Overdrive -> Distortion Pedalboard -> "
    "Amp -> Cab -> EQ -> Reverb"
    "</div>")

button_row = widgets.HBox([b_apply, b_bypass, b_refresh])
preset_row = widgets.HBox(preset_buttons)

dist_box = widgets.VBox([
    widgets.HBox([w_dist_master, w_dist_stack, w_dist_exclusive]),
    w_dist_pedal,
    w_dist_multi,
    w_dist_drive, w_dist_tone, w_dist_level,
    w_dist_bias, w_dist_tight, w_dist_mix,
])

acc = widgets.Accordion(children=[
    widgets.VBox([w_gate_on, w_gate_thr]),
    widgets.VBox([w_od_on, w_od_drive, w_od_tone, w_od_level]),
    dist_box,
    widgets.VBox([w_amp_on, w_amp_gain, w_amp_bass, w_amp_mid,
                  w_amp_treble, w_amp_presence, w_amp_resonance,
                  w_amp_master, w_amp_character]),
    widgets.VBox([w_cab_on, w_cab_mix, w_cab_level, w_cab_model, w_cab_air]),
    widgets.VBox([w_eq_on, w_eq_low, w_eq_mid, w_eq_high]),
    widgets.VBox([w_rev_on, w_rev_decay, w_rev_tone, w_rev_mix]),
])
for i, name in enumerate(["Noise Gate", "Overdrive", "Distortion Pedalboard",
                          "Amp Simulator", "Cab IR", "EQ", "Reverb"]):
    acc.set_title(i, name)
acc.selected_index = 2  # open Distortion Pedalboard by default

display(widgets.VBox([title, button_row, preset_row, status, acc]))
refresh_status()
